In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_North_Campus_DU_Delhi_IMD_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,227.0,115.0,143.0,121.0,118.0,135.0,80.0,NaN,136.0,NaN,NaN,347.0
1,2,336.0,138.0,160.0,123.0,NaN,140.0,56.0,NaN,134.0,NaN,379.0,NaN
2,3,NaN,151.0,124.0,125.0,105.0,138.0,94.0,NaN,135.0,163.0,465.0,297.0
3,4,324.0,NaN,NaN,121.0,95.0,NaN,115.0,NaN,130.0,NaN,NaN,NaN
4,5,330.0,211.0,NaN,118.0,159.0,144.0,NaN,NaN,127.0,213.0,446.0,NaN
5,6,378.0,254.0,NaN,153.0,214.0,147.0,52.0,NaN,110.0,NaN,435.0,280.0
6,7,345.0,284.0,135.0,139.0,137.0,197.0,55.0,107.0,NaN,NaN,378.0,NaN
7,8,374.0,119.0,197.0,NaN,109.0,112.0,42.0,NaN,NaN,NaN,440.0,316.0
8,9,413.0,143.0,92.0,NaN,155.0,124.0,51.0,NaN,102.0,NaN,NaN,310.0
9,10,391.0,132.0,143.0,NaN,187.0,116.0,70.0,118.0,NaN,211.0,274.0,NaN


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,227.000000,115.000000,143.000000,121.00,118.000000,135.00,80.00,105.894737,136.00,207.740741,307.678571,347.000000
1,2,336.000000,138.000000,160.000000,123.00,134.685714,140.00,56.00,105.894737,134.00,207.740741,379.000000,269.782609
2,3,230.766667,151.000000,124.000000,125.00,105.000000,138.00,63.76,105.894737,135.00,163.000000,465.000000,297.000000
3,4,324.000000,165.653846,121.222222,121.00,95.000000,111.72,63.76,105.894737,130.00,207.740741,307.678571,269.782609
4,5,330.000000,211.000000,121.222222,118.00,159.000000,144.00,63.76,105.894737,127.00,213.000000,446.000000,269.782609
5,6,378.000000,254.000000,121.222222,153.00,134.685714,147.00,52.00,105.894737,110.00,207.740741,435.000000,280.000000
6,7,345.000000,165.653846,135.000000,139.00,137.000000,111.72,55.00,107.000000,114.64,207.740741,378.000000,269.782609
7,8,374.000000,119.000000,121.222222,122.56,109.000000,112.00,63.76,105.894737,114.64,207.740741,440.000000,316.000000
8,9,413.000000,143.000000,92.000000,122.56,155.000000,124.00,51.00,105.894737,102.00,207.740741,307.678571,310.000000
9,10,391.000000,132.000000,143.000000,122.56,187.000000,116.00,70.00,118.000000,114.64,211.000000,274.000000,269.782609
